<a href="https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainasif11/flyrankinternship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Warehouse connection ready")

Warehouse connection ready


## 1. Method choice and why

I will use Logistic Regression for this first model.

I chose it because my task is a binary prediction problem: identify whether a content item should receive attention or not. Logistic Regression is simple, fast, and easy to interpret. It gives me a useful model to compare against my Week-4 rule-based baseline.

I do not want to use a more complex model just because it is more complicated. The goal is to see whether a simple trained model can improve on the baseline.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design


I will use March 2026 data and keep the evaluation data separate from the training data.

I will use a stratified train/test split so that both sets contain examples of the two classes. The test set will only be used for the final comparison with my Week-4 baseline.

I will not use future-window information or label-derived features as model inputs.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_engaged_sessions,
        gsc_data_available
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
""").df()

print("Rows:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,gsc_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,True
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,True
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,True
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,True
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,True


In [19]:
df[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]].describe()

,gsc_impressions,gsc_clicks,gsc_avg_position
count,3.611061e+06,3.611061e+06,3.611061e+06
mean,7.772164e+01,2.275874e-01,1.582665e+01
std,2.498747e+02,1.277267e+00,1.985603e+01
min,1.000000e+00,0.000000e+00,0.000000e+00
25%,4.000000e+00,0.000000e+00,3.742120e+00
50%,1.600000e+01,0.000000e+00,7.500000e+00
75%,6.200000e+01,0.000000e+00,2.020000e+01
max,4.008400e+04,2.740000e+02,4.980000e+02


## 3. Train + compare vs my baseline

I will use the same March 2026 data for both the model and the Week-4 baseline.

The model will use search-performance signals that are available at the decision moment. I will keep the target separate from the input features so that the model does not learn from the answer itself.

I will use F1 as the main metric because the classes may not be evenly balanced. I will compare the Logistic Regression result with the Week-4 baseline using the same test set and the same metric.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["target"] = (df["gsc_clicks"] == 0).astype(int)

print(df["target"].value_counts())

target
1    3193080
0     417981
Name: count, dtype: int64


In [21]:
df["target"] = (df["gsc_clicks"] == 0).astype(int)

features = [
    "gsc_impressions",
    "gsc_avg_position"
]

X = df[features].copy()
y = df["target"].copy()

print("Features:", features)
print("X shape:", X.shape)

Features: ['gsc_impressions', 'gsc_avg_position']
X shape: (3611061, 2)


In [22]:
X = X.replace([float("inf"), float("-inf")], pd.NA)

X = X.fillna(X.median())

print("Missing values:")
print(X.isna().sum())

Missing values:
gsc_impressions     0
gsc_avg_position    0
dtype: int64


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 2888848
Test rows: 722213


In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Logistic Regression trained.")

Logistic Regression trained.


In [25]:
model_pred = model.predict(X_test)

print(model_pred[:20])

[1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

model_f1 = f1_score(
    y_test,
    model_pred,
    zero_division=0
)

model_accuracy = accuracy_score(y_test, model_pred)

model_precision = precision_score(
    y_test,
    model_pred,
    zero_division=0
)

model_recall = recall_score(
    y_test,
    model_pred,
    zero_division=0
)

print("Model accuracy:", model_accuracy)
print("Model precision:", model_precision)
print("Model recall:", model_recall)
print("Model F1:", model_f1)

Model accuracy: 0.8972325338923559
Model precision: 0.9049601423538612
Model recall: 0.9874870227381983
Model F1: 0.9444241436735477


In [27]:
baseline_test = df.loc[X_test.index].copy()

baseline_pred = (
    baseline_test["gsc_clicks"] == 0
).astype(int)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

print("W04 baseline F1:", baseline_f1)

W04 baseline F1: 1.0


In [28]:
baseline_test = df.loc[X_test.index].copy()

baseline_pred = (
    baseline_test["gsc_clicks"] == 0
).astype(int)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

print("W04 baseline F1:", baseline_f1)

W04 baseline F1: 1.0


The Logistic Regression model achieved an F1 score of 0.9444, while the Week-4 baseline achieved 1.0000.

However, this comparison has an important limitation. The target used in this notebook was created from `gsc_clicks == 0`, while the Week-4 baseline also directly used zero clicks. Therefore, the perfect baseline score is expected and does not represent a meaningful independent benchmark.

I will not treat the 1.0000 baseline score as evidence that the rule is genuinely better than the model. The result shows that the target and baseline need to be defined independently for a fair modeling comparison.

## 4. Errors and interpretation

The Logistic Regression model achieved an F1 score of 0.9444. Its recall was high at 0.9875, meaning it identified most of the rows belonging to the positive class.

The model made some incorrect predictions, which shows that impressions and average position alone do not perfectly separate the two classes.

The main limitation of this experiment is the target definition. Because the target was created directly from GSC clicks, the Week-4 baseline also becomes a direct match to the target. This makes the current comparison circular.

For that reason, I treat this result as a modeling exercise rather than evidence that Logistic Regression beats the Week-4 baseline. A future version should use an independently defined future-window outcome so that the baseline and model can be compared fairly.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


errors = pd.DataFrame({
    "actual": y_test,
    "predicted": model_pred
})

errors["correct"] = errors["actual"] == errors["predicted"]

print("Correct predictions:", errors["correct"].sum())
print("Incorrect predictions:", (~errors["correct"]).sum())

errors[~errors["correct"]].head(10)

Correct predictions: 647993
Incorrect predictions: 74220


,actual,predicted,correct
2642134,1,0,False
582157,0,1,False
1672305,0,1,False
3160590,0,1,False
2820222,0,1,False
3061540,0,1,False
3418492,0,1,False
234711,0,1,False
311785,0,1,False
922502,0,1,False


In [30]:
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.named_steps["logistic"].coef_[0]
})

coefficients["absolute_coefficient"] = coefficients["coefficient"].abs()

coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

,feature,coefficient,absolute_coefficient
0,gsc_impressions,-1.537580,1.537580
1,gsc_avg_position,0.775645,0.775645


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.